##### Complaint Agent Stream

This notebook streams complaints through the complaint agent for processing

In [ ]:
%pip install -U -qqqq databricks-sdk requests


In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

import os
import sys
sys.path.append(os.path.abspath("../utils"))
from agent_app_client import app_request_context, extract_response_text, resolve_agent_app_name

try:
    _APP_NAME_PARAM = dbutils.widgets.get("COMPLAINT_AGENT_APP_NAME")
except Exception:
    _APP_NAME_PARAM = ""
# Prefer the deploy-time-baked param so the name carries the deploy-time catalog
# even when --params CATALOG disagrees with --var catalog; resolver re-sanitises.
COMPLAINT_AGENT_APP_NAME = resolve_agent_app_name(_APP_NAME_PARAM, CATALOG, "complaint")

_AGENT_APP_CONTEXT = app_request_context(app_name=COMPLAINT_AGENT_APP_NAME, dbutils=dbutils)
COMPLAINT_AGENT_APP_URL = _AGENT_APP_CONTEXT["url"]
COMPLAINT_AGENT_APP_TOKEN = _AGENT_APP_CONTEXT["bearer_token"]
print(f"Complaint agent app: {COMPLAINT_AGENT_APP_NAME} ({COMPLAINT_AGENT_APP_URL})")


In [ ]:
import json
import os
import random

from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
from pyspark.sql.window import Window

import requests

# Per-call timeout in seconds.  Without this, a stalled / scale-to-zero /
# deleted complaint-agent app causes the underlying HTTP socket to hang
# indefinitely (we observed a single stream run hung for 3h47m on a previous
# test session, blocking the cron queue).  30s is generous for a healthy agent
# call (typical latency is 2-5s) but bounds the worst case.
_CALL_TIMEOUT_S = 30

# Per-batch inference cap and checkpoint path.
# Why this exists: previously this stream had no cap — `availableNow=True` would
# drain the entire `raw_complaints` backlog through the agent UDF in one go,
# and a few hundred rows of backlog times 5-30s per call easily blew through
# the 10-min `timeout_seconds` task budget. The cron then dropped subsequent
# ticks (queue.enabled=False) and the backlog kept growing forever.
#
# Sizing the cap (10) against measured app latency:
#   - Observed warm latency on this complaint-agent app is 16-21s/call,
#     not the 2-5s the prior comment assumed.  With the previous cap of 20
#     and a 3-attempt retry loop, a single timed-out call cost up to 90s,
#     and we consistently blew the 600s budget at ~615s (every cron tick).
#   - With MAX=10 and no retries (see process_complaint below):
#       worst case   10 × 30s        = 300s  (~5min slack)
#       typical      10 × 18s        = 180s  (~7min slack)
#   - Refund + support streams use 50 because their agents are faster;
#     do not blindly copy that cap here without re-measuring.
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/complaints/checkpoints/complaint_agent_stream"
# Sized for the 600s task timeout.  The complaint agent measured around 25s
# per call through Apps + Gateway during smoke testing.  5 × 25 ≈ 125s leaves room for
# cold-start jitter, Delta write, and is_first_run() probes; 10 × 25 = 250s
# + ~250s overhead was tipping over the 600s wall once the agent was
# fully healthy.
MAX_INFERENCES_PER_BATCH = 5


def _extract_agent_text(response):
    return extract_response_text(response)


def is_first_run():
    """True when no micro-batch has been committed yet — drives the
    fake-it-till-up backfill path on cold start.

    Looks at the `commits/` subdirectory specifically.  Spark Structured
    Streaming creates `offsets/`, `commits/`, `metadata`, `sources/` *before*
    the first call to `process_batch`, so the older heuristic of
    `os.listdir(CHECKPOINT_PATH) == 0` always returned False on the very first
    batch — silently disabling the fast path and forcing the entire historical
    backlog through the real-inference codepath in a single micro-batch (which
    is what kept blowing past the 10-min task timeout).  A non-hidden file
    under `commits/` is the unambiguous signal that this is not the first run.
    """
    commits_dir = os.path.join(CHECKPOINT_PATH, "commits")
    if not os.path.exists(commits_dir):
        return True
    return not any(
        not f.startswith(".") and not f.startswith("_")
        for f in os.listdir(commits_dir)
    )


# ── Deterministic fallback responses ──────────────────────────────────────────
# Used both during initial backfill (when the checkpoint is empty, so we want
# to drain the entire historical source instantly) and as the per-batch
# overflow path beyond MAX_INFERENCES_PER_BATCH.  Schema matches the real
# agent response shape; values are realistic decision/credit pairs.

fake_responses = [
    {
        "complaint_category": "late_delivery",
        "decision": "credit",
        "credit_amount": 5.0,
        "rationale": "Order delivered after P75 threshold for the location.",
        "customer_response": "Sorry about the late delivery — we've issued a $5.00 credit to your account.",
    },
    {
        "complaint_category": "missing_item",
        "decision": "credit",
        "credit_amount": 7.5,
        "rationale": "Missing-item complaints are auto-credited at item value when verified by order line.",
        "customer_response": "We're sorry an item was missing from your order — we've added a $7.50 credit.",
    },
    {
        "complaint_category": "food_quality",
        "decision": "escalate",
        "credit_amount": 0.0,
        "rationale": "Food-quality complaints route to a human reviewer with kitchen photos.",
        "customer_response": "Thanks for the report — our quality team will follow up within 24 hours.",
    },
    {
        "complaint_category": "wrong_order",
        "decision": "credit",
        "credit_amount": 10.0,
        "rationale": "Wrong-order complaints trigger a same-day credit equal to subtotal cap.",
        "customer_response": "Apologies for the mixup — a $10.00 credit has been applied to your account.",
    },
    {
        "complaint_category": "other",
        "decision": "no_action",
        "credit_amount": 0.0,
        "rationale": "Insufficient signal for automated remediation; no credit.",
        "customer_response": "Thanks for reaching out — we'll look into this and follow up if needed.",
    },
]


def get_fake_response(order_id: str) -> str:
    """Return a deterministic fake response with the order_id filled in.
    Fast path for backfill and overflow rows."""
    resp = random.choice(fake_responses).copy()
    resp["order_id"] = order_id
    return json.dumps(resp)


def process_complaint(complaint_text: str, order_id: str) -> str:
    """Call the complaint agent app for a complaint row."""
    default_response = json.dumps({
        "order_id": order_id,
        "complaint_category": "other",
        "decision": "escalate",
        "credit_amount": None,
        "confidence": None,
        "priority": "standard",
        "rationale": "agent unavailable or did not return valid JSON",
    })

    try:
        http_response = requests.post(
            f"{COMPLAINT_AGENT_APP_URL}/responses",
            headers={
                "Authorization": f"Bearer {COMPLAINT_AGENT_APP_TOKEN}",
                "Content-Type": "application/json",
            },
            json={
                "input": [{
                    "role": "user",
                    "content": f"{complaint_text} (Order ID: {order_id})",
                }]
            },
            timeout=_CALL_TIMEOUT_S,
        )
        http_response.raise_for_status()
        response = _extract_agent_text(http_response.json())
        json.loads(response)
        return response
    except Exception:
        return default_response


process_complaint_udf = udf(process_complaint, StringType())
get_fake_response_udf = udf(get_fake_response, StringType())

In [ ]:
# Stream processing.  Wrapped in a builder so the retry path can pass
# startingVersion="latest" and skip a historical backfill after a table recreate.
# Note: unlike the previous version, we deliberately keep the raw columns here
# and run the agent UDF inside the foreachBatch handler so we can cap the
# number of real inferences per micro-batch (see process_batch).
def _build_complaint_source(starting_version=None):
    reader = spark.readStream
    if starting_version is not None:
        reader = reader.option("startingVersion", str(starting_version))
    return reader.table(f"{CATALOG}.complaints.raw_complaints").select(
        F.col("complaint_id"),
        F.col("order_id"),
        F.col("complaint_text"),
        F.col("ts").alias("source_ts"),
    )


complaint_source = _build_complaint_source()


def process_batch(batch_df, batch_id):
    """Process each micro-batch with inference capping.

    - First run (no checkpoint): ALL rows get fake responses (fast backfill).
    - Subsequent runs: first MAX_INFERENCES_PER_BATCH rows get real agent
      inference, rest get deterministic fakes.  This bounds per-run wall time
      to roughly MAX_INFERENCES_PER_BATCH * worst_case_latency, so a backlog
      cannot blow the 10-min task timeout.
    """
    if batch_df.isEmpty():
        return

    first_run = is_first_run()
    row_count = batch_df.count()

    print(f"Processing batch {batch_id}: {row_count} rows, first_run={first_run}")

    if first_run:
        # Drain the historical backlog with fakes — fast path.
        print(f"  -> First run detected, using fake responses for all {row_count} rows")
        result_df = batch_df.select(
            F.col("complaint_id"),
            F.col("order_id"),
            F.current_timestamp().alias("ts"),
            get_fake_response_udf(F.col("order_id")).alias("agent_response"),
        )
    else:
        # Stable order so the cap is deterministic and resumable.
        windowed = batch_df.withColumn(
            "row_num",
            F.row_number().over(Window.orderBy(F.col("source_ts"), F.col("complaint_id"))),
        )

        real_count = min(row_count, MAX_INFERENCES_PER_BATCH)
        fake_count = max(0, row_count - MAX_INFERENCES_PER_BATCH)
        print(f"  -> Real inference: {real_count} rows, fake: {fake_count} rows")

        real_inference_df = windowed.filter(f"row_num <= {MAX_INFERENCES_PER_BATCH}").select(
            F.col("complaint_id"),
            F.col("order_id"),
            F.current_timestamp().alias("ts"),
            process_complaint_udf(F.col("complaint_text"), F.col("order_id")).alias("agent_response"),
        )

        fake_response_df = windowed.filter(f"row_num > {MAX_INFERENCES_PER_BATCH}").select(
            F.col("complaint_id"),
            F.col("order_id"),
            F.current_timestamp().alias("ts"),
            get_fake_response_udf(F.col("order_id")).alias("agent_response"),
        )

        result_df = real_inference_df.union(fake_response_df)

    result_df.write.mode("append").saveAsTable(f"{CATALOG}.complaints.complaint_responses")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS ${CATALOG}.complaints;
CREATE VOLUME IF NOT EXISTS ${CATALOG}.complaints.checkpoints;

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${CATALOG}.complaints.complaint_responses (
  complaint_id STRING,
  order_id STRING,
  ts TIMESTAMP,
  agent_response STRING
)

In [ ]:
# Enable Change Data Feed for Lakebase sync — idempotent so we don't churn
# the Delta table history with a SET TBLPROPERTIES commit on every cron tick
# (we observed 4+ no-op SET TBLPROPERTIES versions accumulating from this).
table_name = f"{CATALOG}.complaints.complaint_responses"

try:
    props = spark.sql(f"SHOW TBLPROPERTIES {table_name}").collect()
    cdc_enabled = any(row.key == "delta.enableChangeDataFeed" and row.value == "true" for row in props)
except Exception:
    cdc_enabled = False

if not cdc_enabled:
    print(f"Enabling CDC on {table_name}")
    spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
else:
    print(f"CDC already enabled on {table_name}, skipping")

In [ ]:
# Process with foreachBatch so process_batch can apply the per-batch inference
# cap (see cell defining MAX_INFERENCES_PER_BATCH).
def _run_stream(stream_df):
    q = (
        stream_df.writeStream
        .foreachBatch(process_batch)
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()


# Recoverable errors that all warrant clearing the checkpoint and restarting
# from the LATEST version of the source:
#   - DIFFERENT_DELTA_TABLE_READ_BY_STREAMING_SOURCE: source table was recreated.
#   - STREAMING_CHECKPOINT_METADATA_ERROR / MISSING_METADATA_FILE: checkpoint
#     directory was partially wiped (e.g. by `bundle run cleanup`) leaving
#     offsets/commits without the metadata file, which Spark refuses to start on.
_RECOVERABLE_MARKERS = (
    "DIFFERENT_DELTA_TABLE_READ_BY_STREAMING_SOURCE",
    "STREAMING_CHECKPOINT_METADATA_ERROR",
    "MISSING_METADATA_FILE",
)

try:
    _run_stream(complaint_source)
except Exception as e:
    msg = str(e)
    if any(marker in msg for marker in _RECOVERABLE_MARKERS):
        first_line = msg.splitlines()[0] if msg else ""
        print(
            f"⚠️ Streaming state unrecoverable ({first_line}). "
            f"Clearing stale checkpoint at {CHECKPOINT_PATH} and restarting "
            f"from the LATEST version of the source (historical backfill skipped)."
        )
        dbutils.fs.rm(CHECKPOINT_PATH, recurse=True)
        print(f"✅ Cleared {CHECKPOINT_PATH}")
        _run_stream(_build_complaint_source(starting_version="latest"))
    else:
        raise